In [1]:
#import current working directory

import os
import sys

print(os.getcwd())

c:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation\research


In [2]:
# Read the sys path
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation\src")
sys.path.append(r"C:\Users\Greesha Vaishnavi\Desktop\dsprojects\E_Commerce_Customer_Segmentation")

In [3]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\E_Commerce_Customer_Segmentation\\research'

In [4]:
# move to parent directory

os.chdir("../") 

In [5]:
# present working directory

%pwd

'c:\\Users\\Greesha Vaishnavi\\Desktop\\dsprojects\\E_Commerce_Customer_Segmentation'

In [6]:
# test project import

import box

print(box.__version__)

7.4.1


In [7]:
# Entity

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ModelEvaluationConfig:

    root_dir: Path
    transformed_data_file: Path
    trained_model_path: Path
    scaler_path: Path
    report_path: Path

In [8]:
# Import project libraries

from E_Commerce_Customer_Segmentation.constant import *
from E_Commerce_Customer_Segmentation.utils.common import read_yaml,create_directories
from E_Commerce_Customer_Segmentation.entity.config_entity import ModelEvaluationConfig
from E_Commerce_Customer_Segmentation.config.configuration import ConfigurationManager


In [9]:
def get_model_evaluation_config(self) -> ModelEvaluationConfig:

    config = self.config["model_evaluation"]

    create_directories([config.root_dir])

    model_evaluation_config = ModelEvaluationConfig(

        root_dir=Path(config.root_dir),

        transformed_data_file=Path(config.transformed_data_file),

        trained_model_path=Path(config.trained_model_path),

        scaler_path=Path(config.scaler_path),

        report_path=Path(config.report_path)
    )

    return model_evaluation_config

In [10]:
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import silhouette_score,davies_bouldin_score,calinski_harabasz_score

from E_Commerce_Customer_Segmentation.logging import logger
from E_Commerce_Customer_Segmentation.entity.config_entity import ModelEvaluationConfig

In [11]:
# components

class ModelEvaluation:

    def __init__(self, config):

        self.config = config

    # Load transformed RFM data

    def load_data(self):

        rfm = pd.read_csv(
            self.config.transformed_data_file
        )

        logger.info(
            f"RFM data loaded successfully with shape: {rfm.shape}"
        )

        return rfm

    # Load trained model

    def load_model(self):

        model = joblib.load(
            self.config.trained_model_path
        )

        logger.info(
            "Trained model loaded successfully"
        )

        return model

    # Load fitted scaler

    def load_scaler(self):

        scaler = joblib.load(
            self.config.scaler_path
        )

        logger.info(
            "Scaler loaded successfully"
        )

        return scaler

    # Prepare RFM features

    def prepare_data(self, rfm, scaler):

        X = rfm[
            [
                "Recency",
                "Frequency",
                "Monetary"
            ]
        ]

        # Apply same log transformation used during training

        X_log = np.log1p(X)

        # Use saved scaler

        X_scaled = scaler.transform(
            X_log
        )

        X_scaled = pd.DataFrame(
            X_scaled,
            columns=X_log.columns,
            index=X_log.index
        )

        return X_scaled

    # Evaluate final model

    def evaluate_model(self):

        logger.info(
            "Starting Model Evaluation"
        )

        # Load data, model and scaler

        rfm = self.load_data()

        model = self.load_model()

        scaler = self.load_scaler()

        # Prepare data

        X_scaled = self.prepare_data(
            rfm,
            scaler
        )

        # Generate cluster predictions

        labels = model.predict(
            X_scaled
        )

        # Number of clusters

        unique_clusters = np.unique(
            labels
        )

        number_of_clusters = len(
            unique_clusters
        )

        # Calculate clustering metrics

        silhouette = silhouette_score(
            X_scaled,
            labels
        )

        davies_bouldin = davies_bouldin_score(
            X_scaled,
            labels
        )

        calinski_harabasz = calinski_harabasz_score(
            X_scaled,
            labels
        )

        # GMM-specific metrics

        if hasattr(model, "aic"):

            aic = model.aic(
                X_scaled
            )

        else:

            aic = np.nan

        if hasattr(model, "bic"):

            bic = model.bic(
                X_scaled
            )

        else:

            bic = np.nan

        # Create evaluation report

        evaluation_report = pd.DataFrame(
            {
                "Metric": [
                    "Silhouette Score",
                    "Davies-Bouldin Index",
                    "Calinski-Harabasz Score",
                    "AIC",
                    "BIC",
                    "Number of Clusters"
                ],

                "Value": [
                    silhouette,
                    davies_bouldin,
                    calinski_harabasz,
                    aic,
                    bic,
                    number_of_clusters
                ]
            }
        )

        # Display evaluation

        print(
            "\nFinal Model Evaluation:"
        )

        print(
            evaluation_report.round(4)
        )

        # Cluster distribution

        cluster_distribution = (
            pd.Series(labels)
            .value_counts()
            .sort_index()
        )

        # Calculate cluster percentage

        cluster_percentage = (
            cluster_distribution
            / len(labels)
            * 100
        )

        # Create cluster distribution DataFrame

        cluster_distribution_df = pd.DataFrame(
            {
                "CustomerCount": cluster_distribution,
                "Percentage": cluster_percentage
            }
        )

        print(
            "\nCluster Distribution:"
        )

        print(
            cluster_distribution_df.round(2)
        )

        # Create customer cluster profile

        rfm["GMM_Cluster"] = labels

        cluster_profile = (
            rfm
            .groupby("GMM_Cluster")
            .agg(
                Recency=("Recency", "mean"),
                Frequency=("Frequency", "mean"),
                Monetary=("Monetary", "mean"),
                CustomerCount=("CustomerID", "count")
            )
            .reset_index()
        )

        # Calculate cluster percentage

        cluster_profile["Percentage"] = (
            cluster_profile["CustomerCount"]
            / len(rfm)
            * 100
        )

        print(
            "\nCustomer Cluster Profile:"
        )

        print(
            cluster_profile.round(2)
        )

        # Save evaluation report

        evaluation_report.to_csv(
            self.config.report_path,
            index=False
        )

        logger.info(
            f"Evaluation report saved to: {self.config.report_path}"
        )

        # Save cluster profile

        cluster_profile_path = os.path.join(
            self.config.root_dir,
            "cluster_profile.csv"
        )

        cluster_profile.to_csv(
            cluster_profile_path,
            index=False
        )

        logger.info(
            f"Cluster profile saved to: {cluster_profile_path}"
        )

        logger.info(
            "Model Evaluation Completed Successfully"
        )

        return evaluation_report

In [12]:
# Pipeline


class ModelEvaluationTrainingPipeline:

    def __init__(self):
        pass

    def main(self):

        try:

            logger.info(">>>>>> Model Evaluation Stage Started <<<<<<")

            config = ConfigurationManager()

            model_evaluation_config = (config.get_model_evaluation_config())

            model_evaluation = ModelEvaluation(config=model_evaluation_config)

            model_evaluation.evaluate_model()

            logger.info(">>>>>> Model Evaluation Stage Completed Successfully <<<<<<")

        except Exception as e:

            logger.exception(e)

            raise e

In [13]:
obj = ModelEvaluationTrainingPipeline()
obj.main()

[2026-10-01 02:21:42,947: INFO: 1330661342: >>>>>> Model Evaluation Stage Started <<<<<<]
[2026-10-01 02:21:42,956: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-01 02:21:42,960: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-01 02:21:42,963: INFO: common: created directory at: artifacts]
[2026-10-01 02:21:42,966: INFO: common: created directory at: artifacts/model_evaluation]
[2026-10-01 02:21:42,966: INFO: 2088544112: Starting Model Evaluation]
[2026-10-01 02:21:42,973: INFO: 2088544112: RFM data loaded successfully with shape: (4338, 4)]
[2026-10-01 02:21:43,174: INFO: 2088544112: Trained model loaded successfully]
[2026-10-01 02:21:43,189: INFO: 2088544112: Scaler loaded successfully]

Final Model Evaluation:
                    Metric       Value
0         Silhouette Score      0.4307
1     Davies-Bouldin Index      0.9023
2  Calinski-Harabasz Score   4300.7594
3                      AIC  32563.4495
4                      BIC  32620.